# GSK Indication Split Model

This notebook trains an interpretable hospital-level model that splits total 6-month sales across Indications A, B, and C.

The modelling choices are intentionally simple:
- weighted multinomial logistic regression as the champion model
- additive log-ratio OLS as a statistical benchmark
- a compact feature set designed for business interpretation

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from gsk_model import (
    FEATURE_COLUMNS,
    TARGET_COLUMNS,
    build_modeling_frame,
    load_exported_artifacts,
    predict_multinomial,
    prepare_app_features_from_inputs,
    train_and_export,
)

sns.set_theme(style="whitegrid", context="talk")
DATA_PATH = Path("sales.xlsx")

## 1. Load and prepare the hospital-level dataset

The workbook has a three-row header, so the shared module cleans the header once and creates a tidy modelling table.

In [ ]:
modeling_df = build_modeling_frame(DATA_PATH)
display(modeling_df[["hospital_id", "total_6m_sales", *FEATURE_COLUMNS, *TARGET_COLUMNS]].head())

print(f"Hospitals: {len(modeling_df)}")
print(f"Target sum check (max absolute error): {modeling_df['target_sum_error'].max():.8f}")

## 2. Feature set and target summary

We keep only the variables needed to explain the story clearly: hospital size, sales stability, total commercial effort, the commercial mix by indication, and touchpoint intensity per reached HCP.

In [ ]:
display(modeling_df[FEATURE_COLUMNS + TARGET_COLUMNS].describe().T.round(3))

## 3. Train the champion model and benchmark

The training routine uses:
- a fixed 80/20 hospital-level train/test split with `random_state=42`
- 5-fold cross-validation on the training portion to choose a small `C` grid for the multinomial model
- additive log-ratio OLS as a benchmark for comparison

In [ ]:
artifacts = train_and_export(DATA_PATH)

print("Best C:", artifacts.config["best_c"])
display(artifacts.tuning_results)
display(artifacts.cv_metrics.round(4))
display(artifacts.holdout_metrics.round(4))

## 4. Champion vs benchmark comparison

The multinomial model is preferred if it produces lower error while preserving valid predicted shares that sum to one.

In [ ]:
comparison = pd.concat(
    [
        artifacts.cv_metrics.assign(model="Weighted multinomial", split="Cross-validation"),
        artifacts.benchmark_cv_metrics.assign(model="ALR OLS", split="Cross-validation"),
        artifacts.holdout_metrics.assign(model="Weighted multinomial", split="Holdout"),
        artifacts.benchmark_holdout_metrics.assign(model="ALR OLS", split="Holdout"),
    ],
    ignore_index=True,
)
display(comparison.round(4))

## 5. Actual vs predicted plots

These plots make the evaluation easy to read: tighter clustering around the diagonal means more accurate split allocation.

In [ ]:
def plot_actual_vs_predicted(pred_df, title):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, label in zip(axes, ["a", "b", "c"]):
        ax.scatter(
            pred_df[f"avg_split_{label}"],
            pred_df[f"pred_split_{label}"],
            alpha=0.8,
            color={"a": "#12146b", "b": "#ff6a00", "c": "#10a37f"}[label],
        )
        ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
        ax.set_title(f"Indication {label.upper()}")
        ax.set_xlabel("Actual share")
        ax.set_ylabel("Predicted share")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_actual_vs_predicted(artifacts.cv_predictions, "Champion model: cross-validated predictions")
plot_actual_vs_predicted(artifacts.holdout_predictions, "Champion model: holdout predictions")

## 6. Coefficient interpretation

Multinomial logistic regression stays interpretable because each coefficient tells us how a feature shifts the odds toward one indication relative to the others.

In [ ]:
final_model, config = load_exported_artifacts()
coef_df = pd.DataFrame(
    final_model.named_steps["clf"].coef_,
    columns=FEATURE_COLUMNS,
    index=["Indication A", "Indication B", "Indication C"],
)
display(coef_df.round(3))

pd.DataFrame(config["top_drivers"]).T

## 7. Exported artifacts for Streamlit

The notebook writes:
- `artifacts/gsk_multinomial_model.joblib`
- `artifacts/gsk_model_config.json`

The app reuses these artifacts for communication and scenario testing.

In [ ]:
scenario_inputs = {
    "total_6m_sales": 1800,
    "sales_cv": 0.18,
    "total_touchpoints_all": 260,
    "touchpoints_share_a": 0.55,
    "touchpoints_share_b": 0.25,
    "total_hcps_all": 140,
    "hcp_share_a": 0.5,
    "hcp_share_b": 0.3,
}

scenario_features = prepare_app_features_from_inputs(scenario_inputs)
scenario_prediction = predict_multinomial(final_model, scenario_features)
display(scenario_features)
display(scenario_prediction.round(4))